# 04 — Total Replacement Compression

This notebook demonstrates the **total replacement** compression strategy:
compress an entire cached sequence at once by gathering all tokens,
scoring them with KeyDiff, selecting the top-k per head, and scattering
the survivors back into the first `compacted_len` slots.

The cache shrinks — the block table stays the same, but the sequence
length tracked by the scheduler decreases.

Notebooks 01/02 validated the gather-select-scatter primitives using
random token selection. Notebook 03 validated the KeyDiff scoring
primitive. This notebook combines them: real scoring drives the
selection instead of random indices.

We run two experiments:
1. **Compact with KeyDiff scoring** — scatter a sequence, compact it
   using KeyDiff, and verify the result
2. **Score quality check** — confirm that the kept tokens are indeed
   the most distinctive ones

## Imports and Setup

In [ ]:
import torch
import torch.nn.functional as F
from vllm import _custom_ops as ops

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
SEQ_LEN = 137
COMPRESSION_RATIO = 0.5
DEVICE = "cuda"

## Primitives

Cache operations from notebook 02, scoring from notebook 03, and
a `score_and_select` function that combines scoring with top-k
selection to produce `kept_indices` for compaction.

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values


def select_per_head(dense, kept_indices):
    head_size = dense.shape[2]
    by_head = dense.permute(1, 0, 2)
    idx = kept_indices.unsqueeze(-1).expand(-1, -1, head_size)
    kept = by_head.gather(1, idx)
    return kept.permute(1, 0, 2).contiguous()


def compact_kv_cache(key_cache, value_cache, slot_mapping, kept_indices,
                     block_table, block_size,
                     kv_cache_dtype="auto", k_scale=None, v_scale=None):
    device = key_cache.device
    compacted_len = kept_indices.shape[1]

    if k_scale is None:
        k_scale = torch.tensor(1.0, dtype=torch.float32, device=device)
    if v_scale is None:
        v_scale = torch.tensor(1.0, dtype=torch.float32, device=device)

    keys_dense, values_dense = gather_from_paged_cache(
        key_cache, value_cache, slot_mapping, block_size,
    )
    keys_compact = select_per_head(keys_dense, kept_indices)
    values_compact = select_per_head(values_dense, kept_indices)
    del keys_dense, values_dense

    compact_positions = torch.arange(
        compacted_len, dtype=torch.long, device=device,
    )
    compact_slot_mapping = build_slot_mapping_for_positions(
        block_table, compact_positions, block_size,
    )
    ops.reshape_and_cache_flash(
        keys_compact, values_compact,
        key_cache, value_cache,
        compact_slot_mapping, kv_cache_dtype, k_scale, v_scale,
    )

    return compacted_len


print("Cache functions defined")

In [ ]:
def keydiff_score(keys):
    """Score keys using KeyDiff's key-similarity metric.

    keys: [seq_len, num_kv_heads, head_dim]
    Returns: [num_kv_heads, seq_len]. Higher scores = more important.
    """
    keys_by_head = keys.permute(1, 0, 2)
    normalized = F.normalize(keys_by_head, p=2, dim=-1)
    anchor = normalized.mean(dim=1, keepdim=True)
    scores = -F.cosine_similarity(keys_by_head, anchor, dim=-1)
    return scores


def score_and_select(keys, compacted_len):
    """Score keys with KeyDiff and select the top-k per head.

    Returns: [num_kv_heads, compacted_len] — sorted indices per head.
    """
    scores = keydiff_score(keys)
    _, top_indices = scores.topk(compacted_len, dim=-1)
    return top_indices.sort(dim=-1).values


print("Scoring and selection functions defined")

## Experiment 1 — Compact with KeyDiff Scoring

Scatter a sequence into the paged cache, then compact it using KeyDiff
scoring instead of random selection. Verify that the compacted cache
contains exactly the tokens that KeyDiff ranked highest.

In [ ]:
keys_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)
values_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)

num_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
positions = torch.arange(SEQ_LEN, dtype=torch.long, device=DEVICE)
slot_mapping = build_slot_mapping_for_positions(block_table, positions, BLOCK_SIZE)

k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
ops.reshape_and_cache_flash(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

compacted_len = int(SEQ_LEN * (1 - COMPRESSION_RATIO))

print(f"Sequence length:    {SEQ_LEN}")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Compacted length:   {compacted_len}")

In [ ]:
kept_indices = score_and_select(keys_original, compacted_len)

expected_keys = select_per_head(keys_original, kept_indices)
expected_values = select_per_head(values_original, kept_indices)

result_len = compact_kv_cache(
    key_cache, value_cache,
    slot_mapping, kept_indices, block_table,
    BLOCK_SIZE,
    kv_cache_dtype="auto", k_scale=k_scale, v_scale=v_scale,
)

assert result_len == compacted_len

compact_positions = torch.arange(compacted_len, dtype=torch.long, device=DEVICE)
compact_slot_mapping = build_slot_mapping_for_positions(
    block_table, compact_positions, BLOCK_SIZE,
)
keys_after, values_after = gather_from_paged_cache(
    key_cache, value_cache, compact_slot_mapping, BLOCK_SIZE,
)

torch.testing.assert_close(keys_after, expected_keys, atol=0, rtol=0)
torch.testing.assert_close(values_after, expected_values, atol=0, rtol=0)

print(f"Compacted {SEQ_LEN} -> {compacted_len} tokens using KeyDiff scoring")
print(f"Keys and values match expected selection")

## Experiment 2 — Score Quality Check

Confirm that the kept tokens are indeed the most distinctive ones:
their mean KeyDiff score should be higher than the overall mean.

In [ ]:
scores = keydiff_score(keys_original)
mean_kept_score = scores.gather(1, kept_indices).mean().item()
mean_all_score = scores.mean().item()

assert mean_kept_score > mean_all_score

print(f"Mean score (all tokens):  {mean_all_score:.4f}")
print(f"Mean score (kept tokens): {mean_kept_score:.4f}")
print(f"Kept tokens score higher — KeyDiff is selecting distinctive keys")

## Notes and Next Steps

**Total replacement validated.** The full pipeline — scatter → gather →
score with KeyDiff → select top-k per head → scatter back — produces
correct compacted caches. The kept tokens are exactly those that KeyDiff
ranks as most distinctive (dissimilar to the anchor).

**What this enables:** This strategy is suitable for post-prefill
compression: after a long prompt is fully cached, run one compaction
pass to free blocks for other sequences. The scheduler would update
the sequence length to `compacted_len` after compaction.